# Can a protein language model predict where a protein lives in the cell?

**Plan:** turn protein sequences into embeddings with ESM-2 (frozen, no training of the big model), then train a simple classifier on top to predict subcellular localization (nucleus, mitochondrion, etc.).

**Before running:** in Colab, set *Runtime > Change runtime type > T4 GPU* (it also works on CPU with the smallest model, just slower).

Steps: 1) install and load data, 2) embed, 3) classify, 4) visualize, 5) compare model sizes, 6) bonus: masked-residue prediction.

In [ ]:
!pip install -q transformers scikit-learn umap-learn matplotlib pandas

In [ ]:
import os, re, random, urllib.request
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Data: DeepLoc 1.0 dataset

A FASTA file where each header looks like `>Q9SMX3 Mitochondrion-M test`:
UniProt accession, location label (+ `-S` soluble / `-M` membrane / `-U` unknown), and the word `test` if the protein belongs to the held-out test set.

We use the **provided train/test split** instead of a random one. Random splits let near-identical proteins land in both sets, which inflates accuracy.

In [ ]:
URL = "https://services.healthtech.dtu.dk/services/DeepLoc-1.0/deeploc_data.fasta"
if not os.path.exists("deeploc_data.fasta"):
    urllib.request.urlretrieve(URL, "deeploc_data.fasta")

def parse_deeploc(path):
    rows, header, seq = [], None, []
    def flush():
        if header is not None:
            parts = header.split()
            rows.append(dict(
                acc=parts[0],
                label=parts[1].split("-")[0],
                is_test=("test" in parts[2:]),
                seq="".join(seq)))
    for line in open(path):
        line = line.strip()
        if line.startswith(">"):
            flush(); header, seq = line[1:], []
        elif line:
            seq.append(line)
    flush()
    return pd.DataFrame(rows)

df = parse_deeploc("deeploc_data.fasta")
print(df.shape)
print(df.label.value_counts())
print("test fraction:", df.is_test.mean().round(3))

In [ ]:
# Keep the notebook fast: sample a subset (raise these numbers if you have a GPU and time)
MAX_LEN   = 1000     # ESM-2 handles up to 1022 residues; longer sequences are cut
N_TRAIN   = 3000
N_TEST    = 1000

df = df[df.seq.str.len() <= MAX_LEN]
train = df[~df.is_test].sample(min(N_TRAIN, (~df.is_test).sum()), random_state=SEED).reset_index(drop=True)
test  = df[df.is_test].sample(min(N_TEST, df.is_test.sum()), random_state=SEED).reset_index(drop=True)
print(len(train), "train /", len(test), "test")

## 2. Embed with ESM-2

For each protein we get one vector per amino acid, then **average over real residues** (ignoring padding and the start/end tokens) to get a single vector for the whole protein.

In [ ]:
@torch.no_grad()
def embed(seqs, model_name, batch_size=16):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    if device == "cuda":
        model = model.half()
    order = np.argsort([len(s) for s in seqs])        # batch similar lengths together
    out = [None] * len(seqs)
    for i in range(0, len(seqs), batch_size):
        idx = order[i:i + batch_size]
        batch = [seqs[j] for j in idx]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_LEN + 2).to(device)
        hidden = model(**enc).last_hidden_state                       # (B, L, D)
        mask = enc["attention_mask"].clone()
        mask[:, 0] = 0                                                # drop <cls>
        lengths = enc["attention_mask"].sum(1)
        mask[torch.arange(len(batch)), lengths - 1] = 0               # drop <eos>
        mask = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        for j, v in zip(idx, pooled.float().cpu().numpy()):
            out[j] = v
        if (i // batch_size) % 20 == 0:
            print(f"  {i}/{len(seqs)}", end="\r")
    del model
    if device == "cuda": torch.cuda.empty_cache()
    return np.stack(out)

def get_embeddings(model_name):
    tag = model_name.split("/")[-1]
    f = f"emb_{tag}_{len(train)}_{len(test)}.npz"
    if os.path.exists(f):
        z = np.load(f); return z["tr"], z["te"]
    tr = embed(train.seq.tolist(), model_name)
    te = embed(test.seq.tolist(), model_name)
    np.savez(f, tr=tr, te=te)
    return tr, te

MODEL = "facebook/esm2_t6_8M_UR50D"      # smallest: 6 layers, 8M parameters
X_tr, X_te = get_embeddings(MODEL)
print("\nembedding shape:", X_tr.shape)

## 3. Classify

A plain logistic regression on the frozen embeddings. We also compute a **majority-class baseline** so the accuracy means something.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

y_tr, y_te = train.label.values, test.label.values

def fit_eval(X_tr, X_te, C=1.0):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, class_weight="balanced"))
    clf.fit(X_tr, y_tr)
    pred = clf.predict(X_te)
    return clf, pred, accuracy_score(y_te, pred), f1_score(y_te, pred, average="macro")

base = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print("majority-class baseline accuracy:", round(accuracy_score(y_te, base.predict(X_te)), 3))

clf, pred, acc, f1 = fit_eval(X_tr, X_te, C=0.1)
print(f"ESM-2 8M + logistic regression: accuracy={acc:.3f}  macro-F1={f1:.3f}")
print(classification_report(y_te, pred, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(y_te, pred, xticks_rotation=45, ax=ax, colorbar=False)
plt.tight_layout(); plt.show()

For context, DeepLoc 1.0 reported about 78% accuracy on its 10 classes, using a much more elaborate model. Landing somewhat below that with a simple classifier on the smallest ESM-2 is expected.

**Read the confusion matrix:** which compartments get mixed up? Nucleus vs. cytoplasm are the classic confusions, since many proteins shuttle between them.

## 4. Visualize the embedding space

In [ ]:
import umap
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED)
Z = reducer.fit_transform(StandardScaler().fit_transform(X_te))

fig, ax = plt.subplots(figsize=(9, 7))
for lab in sorted(set(y_te)):
    m = y_te == lab
    ax.scatter(Z[m, 0], Z[m, 1], s=8, label=lab, alpha=0.7)
ax.legend(markerscale=2, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_title("ESM-2 8M embeddings of test proteins (UMAP), colored by true location")
plt.tight_layout(); plt.show()

## 5. Does a bigger model help?

Same pipeline, three model sizes. The 150M model needs a GPU to be comfortable (a few minutes on a T4).

In [ ]:
models = {
    "8M":   "facebook/esm2_t6_8M_UR50D",
    "35M":  "facebook/esm2_t12_35M_UR50D",
    "150M": "facebook/esm2_t30_150M_UR50D",
}
results = []
for size, name in models.items():
    print("model", size)
    a, b = get_embeddings(name)
    _, _, acc, f1 = fit_eval(a, b, C=0.1)
    results.append(dict(model=size, accuracy=acc, macro_f1=f1, dim=a.shape[1]))
res = pd.DataFrame(results); display(res)
res.plot(x="model", y=["accuracy", "macro_f1"], kind="bar", ylim=(0, 1), rot=0)
plt.title("Frozen ESM-2 embeddings + logistic regression"); plt.show()

## 6. Bonus: ask the model to fill in blanks

This uses the model's language-model head (the part used in pretraining). Mask one residue in a real protein and see whether ESM-2 guesses it.

In [ ]:
from transformers import EsmForMaskedLM

name = "facebook/esm2_t6_8M_UR50D"
tok = AutoTokenizer.from_pretrained(name)
mlm = EsmForMaskedLM.from_pretrained(name).to(device).eval()

seq = test.seq.iloc[0][:200]
hits, trials = 0, 0
for pos in random.sample(range(len(seq)), 40):
    masked = seq[:pos] + tok.mask_token + seq[pos+1:]
    enc = tok(masked, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = mlm(**enc).logits[0, pos + 1]          # +1 for the <cls> token
    guess = tok.convert_ids_to_tokens(int(logits.argmax()))
    hits += guess == seq[pos]; trials += 1
print(f"guessed the hidden amino acid {hits}/{trials} times ({hits/trials:.0%}); random guessing would be ~5%")